# 03b — Trộn tấn công với nền UNSW, cửa sổ lai theo gói (N gói hoặc T giây)

Giống notebook 03 ở phần chia task và trộn (cùng seed → cùng task/ngày/persona/máy đích), khác ở **cách cắt cửa sổ**:

1. **Cửa sổ lai theo IP nguồn**: các gói host gửi đi được gom liên tiếp; cửa sổ **chốt khi đủ `N_PKT` gói hoặc khi đã `T_MAX` giây**
   kể từ gói đầu cửa sổ, điều kiện nào đến trước. Scan nhanh chốt sau vài chục ms, tấn công chậm vẫn chốt sau 10 s.
2. **Đặc trưng trong cửa sổ** (chỉ tính được khi gom theo gói): số gói, thời lượng, gói/giây, khoảng cách giữa các gói,
   phân bố kích thước gói, tỉ lệ từng loại cờ/giao thức, gói nhận trong cùng khoảng, số đích/cặp/port khác nhau,
   **tỉ lệ đích/cặp/port mới** (host chưa gửi tới trong `NEW_GAP` giây trước).
3. **Ngữ cảnh dài** 60/300 s tính tại thời điểm chốt cửa sổ (đếm, đếm phần tử khác nhau, tỉ lệ) — để bắt host discovery chỉ 6–14 gói.
4. **Nhãn cửa sổ**: `recon`/`bruteforce`/`dos` nếu cửa sổ của `.236` chứa ≥1 gói `attack`; `post_attack`, `victim` không train (như 03).
5. `sample_weight` như 03 (mỗi task tổng 1, kịch bản ∝ √số cửa sổ, cân lớp, trần 0,1%/cửa sổ).

Không dùng IP/MAC/TTL/port cụ thể. Trên router mỗi host chỉ cần vòng đệm gói của cửa sổ hiện tại + bộ đếm theo thời gian cho 60/300 s.

**DoS (v3)**: task DoS từ `01c` (TII-SSRC-23: ICMP/SYN/UDP flood, mỗi phiên tấn công là một task, đã đổi địa chỉ sang
`.236` → `.182` và chia sẵn train/val/test theo thời gian) được trộn vào nền UNSW **y như task lab**.
Thêm nguồn nền `benign_tii` (video/audio/text của TII, cùng cặp máy với DoS) làm nền khó, chiếm `TII_BENIGN_SHARE` trọng số lớp benign.

**Tăng cường DoS (v4)**: dữ liệu DoS chỉ có hping3 của TII (1 port đích, kích thước cố định, nạn nhân luôn trả lời) nên model
học lối tắt ("nạn nhân không trả lời / nhiều port → không phải DoS"). Mỗi task DoS thêm `DOS_AUG` bản biến đổi
(cùng split với task gốc, trộn với ngày nền khác), tham số rút ngẫu nhiên theo seed của bản:
- bỏ phản hồi của nạn nhân: giữ 0% (50%) / 0–50% (25%) / 100% (25%)
- hạ tốc độ (40%): lấy mẫu con xuống 200–3000 gói/s (log-uniform) nếu task nhanh hơn
- rải port đích (50%): TCP 2–10 port cố định; UDP thêm khả năng mỗi gói một port (UDP flood kiểu Mirai).
  Không rải port ngẫu nhiên cho TCP — flood TCP mỗi gói một port trông như quét SYN nhanh, lẫn với recon
- kích thước gói (50%): hằng số 54 B (TCP) / 42 B (ICMP/UDP) → 1500 B (log-uniform), như `hping3 -d`

Đầu ra: `Data_after_processing/windows_pkt/{train,val,test}.csv`, `reports/03b_*.csv`. Notebook 03 (cửa sổ 10 s) giữ nguyên để so sánh.

In [1]:
import zlib
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path('/home/ubuntu/sepcung/03.Brute_Force_vs_Recon')
OUT = ROOT / 'Data_after_processing'
BEN_DIR = OUT / 'benign_packets'
WIN_DIR = OUT / 'windows_pkt'
PART_DIR = WIN_DIR / '_parts'
REPORT_DIR = OUT / 'reports'
TII_BEN_DIR = OUT / 'benign_tii'
WIN_DIR.mkdir(parents=True, exist_ok=True)
PART_DIR.mkdir(parents=True, exist_ok=True)

ATTACKER_IP, ATTACKER_MAC = '192.168.1.236', '50:bb:b5:fe:d8:b8'
TARGET_IP, TARGET_MAC = '192.168.1.182', 'e4:5f:01:a3:c6:0d'
LAN_PREFIX = '192.168.1.'

N_PKT = 50                # chốt cửa sổ khi host gửi đủ số gói này
T_MAX = 10                # ... hoặc khi đã T_MAX giây kể từ gói đầu cửa sổ
CTX = (60, 300)           # giây — ngữ cảnh dài, tính tại thời điểm chốt cửa sổ
NEW_GAP = 300             # giây — đích/cặp/port là "mới" nếu host chưa gửi tới trong NEW_GAP giây trước
HOP = 10                  # chỉ dùng để gộp báo động theo khối 10 s khi báo cáo
PAD = 300                 # đệm nền trước/sau tấn công (>= ngữ cảnh dài nhất)
PERSONA_PROB = 0.8
MIN_DEV_PKTS = 20
SEED = 42
N_JOBS = 12
CLASS_WEIGHT = {'benign': 1.0, 'recon': 1.0, 'bruteforce': 1.0, 'dos': 1.0}
TII_BENIGN_SHARE = 0.1   # tổng trọng số cửa sổ benign TII / tổng trọng số lớp benign (train, val)
MAX_WINDOW_SHARE = 0.001
DOS_AUG = 3                # số bản tăng cường cho mỗi task DoS (0 = tắt, như v3)

TEST_ONLY = ['icmp_echo_discovery', 'tcp_ack_host_discovery', 'tcp_syn_host_discovery']
TRAIN_ONLY_LOTO = ['arp_discovery']

CLIENT_MACS = ['74:2f:68:81:69:42', 'ac:bc:32:d4:6f:2f', 'b4:ce:f6:a7:a3:c2', '40:f3:08:ff:1e:da',
               'd0:a6:37:df:a1:e1', 'f4:5c:89:93:cc:85', '08:21:ef:3b:fc:e3']
CAMERA_MACS = ['70:ee:50:18:34:43', 'f4:f2:6d:93:51:f1', '00:16:6c:ab:6b:88', '30:8c:fb:2f:e4:b2',
               '00:62:6e:51:27:2e', 'e8:ab:fa:19:de:4f', '30:8c:fb:b6:ea:45', '00:24:e4:11:18:a8']

task = pd.read_csv(REPORT_DIR / '01_task_inventory.csv')
days = pd.read_csv(REPORT_DIR / '02_day_summary.csv')
task = task[task['usable']].copy()
print(len(task), 'task dùng được;', len(days), 'ngày nền')

1264 task dùng được; 20 ngày nền


## 1. Chia task tấn công (giống 03)

In [2]:
task['strat'] = task['scenario'] + '|' + task['tool'].fillna('')
task['split'] = ''
task.loc[task['scenario'].isin(TEST_ONLY), 'split'] = 'test'
task.loc[task['scenario'].isin(TRAIN_ONLY_LOTO), 'split'] = 'train'
rest = task[task['split'] == ''].copy()
cnt = rest['strat'].map(rest['strat'].value_counts())
rest.loc[cnt < 7, 'strat'] = rest.loc[cnt < 7, 'scenario']
tr, tmp = train_test_split(rest.index, test_size=0.30, stratify=rest['strat'], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=rest.loc[tmp, 'strat'], random_state=SEED)
task.loc[tr, 'split'], task.loc[va, 'split'], task.loc[te, 'split'] = 'train', 'val', 'test'
task['incomplete_capture'] = task['scenario'].isin(TEST_ONLY).astype(int)

split_tab = task.pivot_table(index='scenario', columns='split', values='task_id', aggfunc='count', fill_value=0)
assert split_tab.equals(pd.read_csv(REPORT_DIR / '03_attack_task_split.csv', index_col='scenario')[split_tab.columns]), \
    'chia task khác notebook 03'

# DoS (01c): đã chia train/val/test theo thời gian trong từng loại — không chia lại
dos_task = pd.read_csv(REPORT_DIR / '01c_dos_task_inventory.csv')
dos_task['incomplete_capture'] = 0
task = pd.concat([task, dos_task], ignore_index=True)
task.pivot_table(index='scenario', columns='split', values='task_id', aggfunc='count', fill_value=0)

split,test,train,val
scenario,,,
arp_discovery,0,2,0
dos_ack,2,12,2
dos_fin,2,11,2
dos_icmp,1,4,1
dos_rst,2,12,2
dos_syn,2,12,2
dos_udp,2,8,2
firewall_ack_mapping,38,176,38
icmp_echo_discovery,118,0,0


## 2. Hàm cửa sổ lai và đặc trưng

- Gói **gửi** của host (theo `ip_src`, hoặc `arp_spa` với ARP) được sắp theo thời gian và cắt tham lam:
  cửa sổ mở ở gói đầu tiên, gom tiếp tới khi đủ `N_PKT` gói hoặc gói kế tiếp rơi ra ngoài `T_MAX` giây.
  Thời điểm chốt = gói thứ `N_PKT` (chốt theo số gói) hoặc gói đầu + `T_MAX` (chốt theo thời gian).
- Gói **nhận** của host trong `[đầu cửa sổ, lúc chốt]` → tỉ lệ RST/SYN-ACK/ICMP unreachable nhận về trên mỗi gói gửi.
- Ngữ cảnh 60/300 s trong `(lúc chốt − S, lúc chốt]`: tổng cộng dồn + `searchsorted` theo khoá `host<<40 | micro giây`;
  số phần tử khác nhau bằng quét khoảng phủ +1/−1 (như 03, nhưng thời gian liên tục thay cho bucket).
- "Mới": gói tới đích/cặp/port mà host chưa gửi tới trong `NEW_GAP` giây trước — trên router chỉ cần bảng last-seen.

In [3]:
FLAGS = ['tcp', 'syn', 'synack', 'rst', 'fin_only', 'null', 'xmas', 'ack_only', 'udp', 'icmp_echo', 'arp_req', 'arp_rep']
CNT_TX = ['pkts', 'bytes'] + FLAGS
CNT_RX = ['rx_pkts', 'rx_rst', 'rx_synack', 'rx_icmp_unreach']
DISTINCT = ['d_dst', 'd_pair', 'd_port', 'd_syn_pair', 'd_arp', 'd_icmp']
RATIOS = ['syn_ratio', 'rx_rst_ratio', 'rx_unreach_ratio', 'syn_per_pair', 'port_per_dst', 'bytes_per_pkt']
NEW = ['dst', 'pair', 'port']
W_FEATS = (['w_pkts', 'w_by_count', 'w_dur', 'w_span', 'w_pps', 'w_iat_mean', 'w_iat_std', 'w_iat_max',
            'w_len_mean', 'w_len_std', 'w_len_min', 'w_len_max']
           + [f'w_r_{c}' for c in FLAGS]
           + ['w_rx_ratio', 'w_rx_rst_ratio', 'w_rx_synack_ratio', 'w_rx_unreach_ratio']
           + [f'w_{d}' for d in DISTINCT] + [f'w_new_{x}_ratio' for x in NEW])
FEATURES = ['f_' + c for c in W_FEATS] + [f'f_{c}_{S}' for S in CTX for c in CNT_TX + CNT_RX + DISTINCT + RATIOS]
US = 1_000_000
SHIFT = np.int64(1) << 40
print(len(FEATURES), 'đặc trưng')


def _col(s, fill=-1):
    return pd.to_numeric(s, errors='coerce').astype('float64').fillna(fill).to_numpy().astype(np.int64)


def _str(s):
    return s.replace('', pd.NA)


def packet_table(pk):
    f = _col(pk['tcp_flags'])
    proto = _col(pk['ip_proto'])
    icmp_t = _col(pk['icmp_type'])
    arp_op = _col(pk['arp_op'], 0)
    tcp_len = _col(pk['tcp_len'], 0)
    dport = np.where(proto == 6, _col(pk['tcp_dport']), np.where(proto == 17, _col(pk['udp_dport']), -1))
    tcp, udp, icmp = proto == 6, proto == 17, proto == 1
    # gói TCP không có cờ = mảnh IP không mang header TCP (vd hping3 -d payload lớn): vẫn là TCP nhưng không bật cờ nào
    # (trước đây cờ NA -> -1 = bật mọi bit -> mỗi mảnh bị đếm thành SYN-ACK + RST + XMAS)
    hdr = tcp & (f >= 0)
    fv = np.where(hdr, f, -1)
    return pd.DataFrame({
        # notebook 01 lưu ô trống là '' -> coi như NA để gói ARP lấy IP từ arp_spa/arp_tpa
        'src': _str(pk['ip_src']).fillna(_str(pk['arp_spa'])).to_numpy(object),
        'dst': _str(pk['ip_dst']).fillna(_str(pk['arp_tpa'])).to_numpy(object),
        'ts': pk['ts'].to_numpy(float),
        'len': _col(pk['len'], 0), 'dport': dport,
        'tcp': tcp, 'syn': hdr & ((fv & 0x12) == 0x02), 'synack': hdr & ((fv & 0x12) == 0x12),
        'rst': hdr & ((fv & 0x04) > 0), 'fin_only': hdr & ((fv & 0x17) == 0x01), 'null': hdr & (fv == 0),
        'xmas': hdr & ((fv & 0x29) == 0x29), 'ack_only': hdr & (fv == 0x10) & (tcp_len == 0),
        'udp': udp, 'icmp_echo': icmp & (icmp_t == 8), 'icmp_unreach': icmp & (icmp_t == 3),
        'arp_req': arp_op == 1, 'arp_rep': arp_op == 2,
        'attack': (pk['label'] == 'attack').to_numpy(),
    })


def _assign_windows(h, tt):
    # h, tt đã sắp theo (host, thời gian); trả về mã cửa sổ tăng dần cho từng gói
    wid = np.empty(len(h), np.int64)
    bounds = np.flatnonzero(np.r_[True, h[1:] != h[:-1], True])
    k = 0
    for a, e in zip(bounds[:-1], bounds[1:]):
        seg = tt[a:e]
        i, n = 0, e - a
        while i < n:
            j = min(i + N_PKT, int(np.searchsorted(seg, seg[i] + T_MAX * US, side='left')))
            wid[a + i:a + j] = k
            k += 1
            i = j
    return wid


def _ctx_distinct(h, item, tt, span, qkeys):
    # số item khác nhau host đã gửi tới trong (q - span, q]
    if len(h) == 0:
        return np.zeros(len(qkeys))
    o = np.lexsort((tt, item, h))
    h, item, tt = h[o], item[o], tt[o]
    new = np.r_[True, (h[1:] != h[:-1]) | (item[1:] != item[:-1]) | (tt[1:] > tt[:-1] + span)]
    st = np.flatnonzero(new)
    last = np.maximum.reduceat(tt, st)
    hh = h[st]
    ev_k = np.r_[hh * SHIFT + tt[st], hh * SHIFT + last + span]
    ev_d = np.r_[np.ones(len(st)), -np.ones(len(st))]
    o = np.argsort(ev_k, kind='stable')
    ev_k, cs = ev_k[o], np.cumsum(ev_d[o])
    idx = np.searchsorted(ev_k, qkeys, side='right') - 1
    return np.where(idx >= 0, cs[np.maximum(idx, 0)], 0)


def window_features(pk, keep_hosts=None, lan_only=False):
    t = packet_table(pk)
    t_base = np.floor(t['ts'].min())
    tt_all = np.round((t['ts'].to_numpy() - t_base) * US).astype(np.int64)
    hosts, inv = np.unique(np.r_[t['src'].fillna('').to_numpy(str), t['dst'].fillna('').to_numpy(str)], return_inverse=True)
    n = len(t)
    hs_all, hd_all = inv[:n].astype(np.int64), inv[n:].astype(np.int64)
    ok = hosts != ''
    if keep_hosts is not None:
        ok &= np.isin(hosts, list(keep_hosts))
    if lan_only:
        ok &= np.char.startswith(hosts, LAN_PREFIX)

    # gói gửi (tx) / nhận (rx) của các host cần đánh giá, sắp theo (host, thời gian)
    m = ok[hs_all]
    tx_idx = np.flatnonzero(m)[np.lexsort((tt_all[m], hs_all[m]))]
    m = ok[hd_all]
    rx_idx = np.flatnonzero(m)[np.lexsort((tt_all[m], hd_all[m]))]
    tx, rx = t.iloc[tx_idx].reset_index(drop=True), t.iloc[rx_idx].reset_index(drop=True)
    h, tt = hs_all[tx_idx], tt_all[tx_idx]
    tx_keys, rx_keys = h * SHIFT + tt, hd_all[rx_idx] * SHIFT + tt_all[rx_idx]
    assert len(tx), 'không có gói gửi của host cần đánh giá'

    wid = _assign_windows(h, tt)
    nW = int(wid[-1]) + 1
    st = np.flatnonzero(np.r_[True, wid[1:] != wid[:-1]])
    en = np.r_[st[1:], len(wid)] - 1
    S = lambda v: np.bincount(wid, weights=np.asarray(v, float), minlength=nW)
    cnt = S(np.ones(len(wid)))
    t_first, t_last = tt[st], tt[en]
    by_count = cnt >= N_PKT
    t_end = np.where(by_count, t_last, t_first + T_MAX * US)
    hw = h[st]
    f = {}

    # --- trong cửa sổ ---
    dur = (t_end - t_first) / US
    f['w_pkts'], f['w_by_count'], f['w_dur'] = cnt, by_count.astype(float), dur
    f['w_span'], f['w_pps'] = (t_last - t_first) / US, cnt / np.maximum(dur, 1e-3)
    same = wid[1:] == wid[:-1]
    iat, iw = np.diff(tt)[same] / US, wid[1:][same]
    n_iat = np.maximum(np.bincount(iw, minlength=nW), 1)
    mean = np.bincount(iw, iat, nW) / n_iat
    f['w_iat_mean'] = mean
    f['w_iat_std'] = np.sqrt(np.maximum(np.bincount(iw, iat ** 2, nW) / n_iat - mean ** 2, 0))
    mx = np.zeros(nW)
    np.maximum.at(mx, iw, iat)
    f['w_iat_max'] = mx
    ln = tx['len'].to_numpy(float)
    lm = S(ln) / cnt
    f['w_len_mean'], f['w_len_std'] = lm, np.sqrt(np.maximum(S(ln ** 2) / cnt - lm ** 2, 0))
    f['w_len_min'], f['w_len_max'] = np.minimum.reduceat(ln, st), np.maximum.reduceat(ln, st)
    for c in FLAGS:
        f[f'w_r_{c}'] = S(tx[c]) / cnt
    rvals = np.c_[np.ones(len(rx)), rx[['rst', 'synack', 'icmp_unreach', 'attack']].to_numpy(float)]
    rcs = np.vstack([np.zeros((1, 5)), np.cumsum(rvals, axis=0)])
    rw = (rcs[np.searchsorted(rx_keys, hw * SHIFT + t_end, side='right')]
          - rcs[np.searchsorted(rx_keys, hw * SHIFT + t_first, side='left')])
    for i, c in enumerate(['w_rx_ratio', 'w_rx_rst_ratio', 'w_rx_synack_ratio', 'w_rx_unreach_ratio']):
        f[c] = rw[:, i] / cnt

    dst = tx['dst'].fillna('').to_numpy(str)
    dstc = pd.factorize(dst)[0].astype(np.int64)
    dport = tx['dport'].to_numpy(np.int64)
    pair = dstc * 65536 + dport
    items = {
        'd_dst': (dst != '', dstc), 'd_pair': (dport >= 0, pair), 'd_port': (dport >= 0, dport),
        'd_syn_pair': (tx['syn'].to_numpy(bool), pair),
        'd_arp': (tx['arp_req'].to_numpy(bool) & (dst != ''), dstc),
        'd_icmp': (tx['icmp_echo'].to_numpy(bool), dstc),
    }
    for name, (mk, it) in items.items():
        f[f'w_{name}'] = np.bincount(np.unique(wid[mk] * (np.int64(1) << 32) + it[mk]) >> 32, minlength=nW).astype(float)
    for x, name in zip(NEW, ['d_dst', 'd_pair', 'd_port']):
        mk, it = items[name]
        idx = np.flatnonzero(mk)
        o = idx[np.lexsort((tt[idx], it[idx], h[idx]))]
        first = np.r_[True, (h[o][1:] != h[o][:-1]) | (it[o][1:] != it[o][:-1]) | (np.diff(tt[o]) > NEW_GAP * US)]
        f[f'w_new_{x}_ratio'] = np.bincount(wid[o], weights=first.astype(float), minlength=nW) / cnt

    # --- ngữ cảnh dài tại lúc chốt ---
    tvals = np.c_[np.ones(len(tx)), ln, tx[FLAGS].to_numpy(float), tx['attack'].to_numpy(float)]
    tcs = np.vstack([np.zeros((1, tvals.shape[1])), np.cumsum(tvals, axis=0)])
    qk = hw * SHIFT + t_end
    hi_t, hi_r = np.searchsorted(tx_keys, qk, side='right'), np.searchsorted(rx_keys, qk, side='right')
    for Sx in CTX:
        lo = hw * SHIFT + np.maximum(t_end - Sx * US, -1)  # không tràn sang khoá của host trước
        ct = tcs[hi_t] - tcs[np.searchsorted(tx_keys, lo, side='right')]
        cr = rcs[hi_r] - rcs[np.searchsorted(rx_keys, lo, side='right')]
        for i, c in enumerate(CNT_TX):
            f[f'{c}_{Sx}'] = ct[:, i]
        for i, c in enumerate(CNT_RX):
            f[f'{c}_{Sx}'] = cr[:, i]
        for name, (mk, it) in items.items():
            f[f'{name}_{Sx}'] = _ctx_distinct(h[mk], it[mk], tt[mk], Sx * US, qk)
        g = lambda c: np.maximum(f[f'{c}_{Sx}'], 1)
        f[f'syn_ratio_{Sx}'] = f[f'syn_{Sx}'] / g('tcp')
        f[f'rx_rst_ratio_{Sx}'] = f[f'rx_rst_{Sx}'] / g('pkts')
        f[f'rx_unreach_ratio_{Sx}'] = f[f'rx_icmp_unreach_{Sx}'] / g('pkts')
        f[f'syn_per_pair_{Sx}'] = f[f'syn_{Sx}'] / g('d_syn_pair')
        f[f'port_per_dst_{Sx}'] = f[f'd_port_{Sx}'] / g('d_dst')
        f[f'bytes_per_pkt_{Sx}'] = f[f'bytes_{Sx}'] / g('pkts')
    attack_trail = ct[:, -1] + cr[:, 4]  # ct/cr của CTX[-1] = 300 s

    out = pd.DataFrame({'host': hosts[hw], 't_start': t_base + t_first / US, 't_end': t_base + t_end / US})
    out['b10'] = np.floor(out['t_end'] / HOP).astype(np.int64)
    out['attack_tx_cur'], out['attack_trail'] = S(tx['attack']), attack_trail
    feats = pd.DataFrame({'f_' + k: v for k, v in f.items()})[FEATURES]
    return pd.concat([out, feats], axis=1)

97 đặc trưng


In [4]:
# Kiểm thử nhỏ
# A: quét SYN 120 port, 100 gói/s -> 3 cửa sổ 50/50/20 (2 chốt theo số gói, 1 chốt theo thời gian)
# B: 3 gói ACK tới cùng 1 cặp tại t=0, 5, 12 -> 2 cửa sổ (2 gói, 1 gói); chỉ gói đầu là "cặp mới"
_a = pd.DataFrame({'ts': np.arange(120) * 0.01, 'ip_src': '10.0.0.1', 'ip_dst': '10.0.0.2',
                   'tcp_dport': np.arange(120), 'tcp_flags': 0x02})
_b = pd.DataFrame({'ts': [0.0, 5.0, 12.0], 'ip_src': '10.0.0.3', 'ip_dst': '10.0.0.9', 'tcp_dport': 80, 'tcp_flags': 0x10})
_toy = pd.concat([_a, _b], ignore_index=True).sort_values('ts', kind='stable').assign(
    len=60, eth_src='a', eth_dst='b', ip_proto=6, ttl=64, tcp_sport=40000, tcp_len=0, udp_sport=pd.NA, udp_dport=pd.NA,
    icmp_type=pd.NA, arp_op=pd.NA, arp_sha=pd.NA, arp_spa=pd.NA, arp_tpa=pd.NA, label='attack')
_w = window_features(_toy)
wa, wb = _w[_w['host'] == '10.0.0.1'], _w[_w['host'] == '10.0.0.3']
assert wa['f_w_pkts'].tolist() == [50, 50, 20] and wa['f_w_by_count'].tolist() == [1, 1, 0]
assert wa['f_w_d_port'].tolist() == [50, 50, 20] and wa['f_d_port_60'].tolist() == [50, 100, 120]
assert wa['f_w_new_port_ratio'].tolist() == [1, 1, 1] and np.allclose(wa['f_w_iat_mean'], 0.01)
assert wb['f_w_pkts'].tolist() == [2, 1] and wb['f_w_dur'].tolist() == [10, 10]
assert wb['f_w_new_pair_ratio'].tolist() == [0.5, 0] and wb['f_d_pair_300'].tolist() == [1, 1]
_w[['host', 't_start', 't_end', 'f_w_pkts', 'f_w_by_count', 'f_w_pps', 'f_w_d_port', 'f_d_port_60', 'f_w_new_pair_ratio']]

,host,t_start,t_end,f_w_pkts,f_w_by_count,f_w_pps,f_w_d_port,f_d_port_60,f_w_new_pair_ratio
0,10.0.0.1,0.0,0.49,50.0,1.0,102.040816,50.0,50.0,1.0
1,10.0.0.1,0.5,0.99,50.0,1.0,102.040816,50.0,100.0,1.0
2,10.0.0.1,1.0,11.00,20.0,0.0,2.000000,20.0,120.0,1.0
3,10.0.0.3,0.0,10.00,2.0,0.0,0.200000,1.0,1.0,0.5
4,10.0.0.3,12.0,22.00,1.0,0.0,0.100000,1.0,1.0,0.0


## 3. Cửa sổ nền: toàn bộ 20 ngày (chỉ host LAN)

In [5]:
DAY_T = days.set_index('day')[['split', 't0', 't1']].to_dict('index')


def day_windows(day):
    pk = pd.read_parquet(BEN_DIR / f'{day}.parquet')
    w = window_features(pk, lan_only=True)
    w.insert(0, 'split', DAY_T[day]['split'])
    w.insert(1, 'source', 'benign_day')
    w.insert(2, 'day', day)
    w['sample_id'] = 'day_' + day
    w['label'] = 'benign'
    w.to_parquet(PART_DIR / f'day_{day}.parquet', index=False)
    return day, len(w), round(w['f_w_by_count'].mean(), 4)


with ProcessPoolExecutor(max_workers=10) as ex:
    day_res = list(ex.map(day_windows, days['day']))
pd.DataFrame(day_res, columns=['day', 'windows', 'share_by_count']).set_index('day').T

day,16-09-23,16-09-24,16-09-25,16-09-26,16-09-27,16-09-28,16-09-29,16-09-30,16-10-01,16-10-02,16-10-03,16-10-04,16-10-05,16-10-06,16-10-07,16-10-08,16-10-09,16-10-10,16-10-11,16-10-12
windows,61638.000,57528.0000,56738.0000,56620.0000,55946.0000,77937.0000,71840.0000,72028.0000,71386.0000,67736.0000,70570.0000,74884.0000,78513.0000,65640.0000,70641.000,57790.0000,56276.0000,56974.0000,69057.0000,64606.0000
share_by_count,0.043,0.0394,0.0106,0.0105,0.0022,0.1582,0.0152,0.0268,0.0123,0.0084,0.0076,0.0874,0.0934,0.0176,0.084,0.0085,0.0069,0.0056,0.1485,0.4758


## 3b. Cửa sổ nền benign TII (01c, mỗi file đã cắt train/val/test; mọi host LAN, không trộn)

In [6]:
tii_parts = pd.read_csv(REPORT_DIR / '01c_tii_benign_parts.csv')


def tii_windows(r):
    sid = f"tii_{r['name']}_{r['split']}"
    try:
        w = window_features(pd.read_parquet(r['parquet']).assign(label=''), lan_only=True)
    except AssertionError:  # đoạn không có gói nào do host LAN gửi
        return sid, 0, np.nan
    w.insert(0, 'split', r['split'])
    w.insert(1, 'source', 'benign_tii')
    w.insert(2, 'day', r['name'])
    w['sample_id'] = sid
    w['label'] = 'benign'
    w.to_parquet(PART_DIR / f'{sid}.parquet', index=False)
    return sid, len(w), round(w['f_w_by_count'].mean(), 4)


with ProcessPoolExecutor(max_workers=6) as ex:
    tii_res = list(ex.map(tii_windows, tii_parts.to_dict('records')))
pd.DataFrame(tii_res, columns=['sample_id', 'windows', 'share_by_count']).set_index('sample_id').T

sample_id,tii_video_http_train,tii_video_http_val,tii_video_http_test,tii_video_rtp_train,tii_video_rtp_val,tii_video_rtp_test,tii_video_udp_train,tii_video_udp_val,tii_video_udp_test,tii_audio_train,tii_audio_val,tii_audio_test,tii_text_train,tii_text_val,tii_text_test,tii_background_train,tii_background_val,tii_background_test
windows,11311.0000,2423.0000,2424.0000,8243.0000,1768.0000,1762.000,7782.0000,1668.0000,1667.0000,10497.0000,2250.0000,2250.0000,729.0000,158.0000,152.0,7.0,2.0,0.0
share_by_count,0.9982,0.9983,0.9983,0.9763,0.9751,0.979,0.9983,0.9976,0.9982,0.9986,0.9987,0.9987,0.0041,0.0127,0.0,0.0,0.0,NaN


## 4. Mẫu trộn: mỗi task tấn công + một đoạn nền cùng split (giống 03)

Task DoS có thêm `DOS_AUG` mẫu `<sample_id>_augK`: gói tấn công được biến đổi bằng `augment_dos` (xem đầu notebook)
trước khi trộn; phần trộn (đoạn nền, persona, máy đích) giữ nguyên logic.

In [7]:
def _remap(df, ip_old, mac_old, ip_new, mac_new):
    for c in ['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa']:
        df[c] = df[c].mask(df[c] == ip_old, ip_new)
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].mask(df[c] == mac_old, mac_new)
    return df


def _pick(ben, macs, rng):
    lan = ben[ben['eth_src'].isin(macs) & ben['ip_src'].fillna('').str.startswith(LAN_PREFIX)]
    vc = lan['eth_src'].value_counts()
    vc = vc[vc >= MIN_DEV_PKTS]
    if vc.empty:
        return None, None
    mac = rng.choice(vc.index.to_numpy())
    ip = lan.loc[lan['eth_src'] == mac, 'ip_src'].mode().iat[0]
    return mac, ip


def augment_dos(d, rng, scen):
    """Biến thể DoS: d = gói attack/aux của task (đã đổi sang ATTACKER_IP -> TARGET_IP). Trả về (gói mới, mô tả)."""
    att = d['label'] == 'attack'
    fwd = att & (d['ip_src'] == ATTACKER_IP)
    rev = att & (d['ip_src'] == TARGET_IP)
    keep = np.ones(len(d), bool)
    desc = {}
    # tốc độ: lấy mẫu con cả hai chiều xuống T gói/s
    sec = np.floor(d.loc[fwd, 'ts']).astype(np.int64)
    pps = sec.value_counts().median() if fwd.any() else 0
    if rng.random() < 0.4:
        T = float(np.exp(rng.uniform(np.log(200), np.log(3000))))
        if pps > T:
            keep &= ~(att.to_numpy() & (rng.random(len(d)) > T / pps))
            desc['rate'] = round(T)
    desc.setdefault('rate', 'orig')
    # phản hồi của nạn nhân (nạn nhân quá tải / tường lửa chặn / chỉ bắt được một chiều)
    u = rng.random()
    rk = 0.0 if u < 0.5 else (rng.uniform(0, 0.5) if u < 0.75 else 1.0)
    keep &= ~(rev.to_numpy() & (rng.random(len(d)) >= rk))
    desc['resp_keep'] = round(rk, 2)
    d = d[keep].copy()
    # rải port đích: k port cố định; UDP có thể mỗi gói một port (TCP thì không — trông như quét SYN nhanh)
    if rng.random() < 0.5:
        f = (d['label'] == 'attack') & (d['ip_src'] == ATTACKER_IP)
        n = int(f.sum())
        if scen == 'dos_udp' and rng.random() < 0.5:
            ports = rng.integers(1, 65536, n)
            desc['ports'] = 'rand'
        else:
            k = int(rng.integers(2, 11))
            ports = rng.choice(rng.integers(1, 65536, k), n)
            desc['ports'] = k
        for c in ['tcp_dport', 'udp_dport']:
            m = f & d[c].notna()  # mảnh IP không có port -> giữ nguyên
            d.loc[m, c] = ports[m[f].to_numpy()]
    desc.setdefault('ports', 1)
    # kích thước gói tấn công (hping3 -d): từ kích thước tối thiểu thật 54 B TCP / 42 B ICMP-UDP
    if rng.random() < 0.5:
        lo = 54 if scen in ('dos_syn', 'dos_ack', 'dos_fin', 'dos_rst') else 42
        L = int(np.exp(rng.uniform(np.log(lo), np.log(1500))))
        f = (d['label'] == 'attack') & (d['ip_src'] == ATTACKER_IP)
        d.loc[f, 'len'] = L
        d.loc[f & (d['ip_proto'] == 6), 'tcp_len'] = max(L - 54, 0)
        desc['len'] = L
    else:
        desc['len'] = 'orig'
    return d, desc


def build_sample(job):
    rng = np.random.default_rng(job['seed'])
    att = pd.read_parquet(job['parquet'])
    att = att[att['label'].isin(['attack', 'aux'])].copy()
    aug = {}
    if job.get('aug'):  # biến thể DoS: rng riêng cùng seed, phần trộn bên dưới giữ nguyên
        att, aug = augment_dos(att, np.random.default_rng(job['seed']), job['scenario'])
        a = att.loc[att['label'] == 'attack', 'ts']
        job = {**job, 'attack_t0': a.min(), 'attack_t1': a.max()}
    a0, a1 = job['attack_t0'], job['attack_t1']
    seg_len = (a1 - a0) + 2 * PAD
    d0, d1 = DAY_T[job['day']]['t0'], DAY_T[job['day']]['t1']
    seg0 = rng.uniform(d0, max(d0, d1 - seg_len))
    att['ts'] = att['ts'] + (seg0 + PAD - a0)
    ben = pd.read_parquet(BEN_DIR / f"{job['day']}.parquet",
                          filters=[('ts', '>=', seg0), ('ts', '<', seg0 + seg_len)])
    persona_mac = persona_ip = None
    if rng.random() < PERSONA_PROB:
        persona_mac, persona_ip = _pick(ben, CLIENT_MACS, rng)
    target_mac, target_ip = _pick(ben, CAMERA_MACS, rng)
    if persona_mac:
        ben = _remap(ben, persona_ip, persona_mac, ATTACKER_IP, ATTACKER_MAC)
    if target_mac:
        ben = _remap(ben, target_ip, target_mac, TARGET_IP, TARGET_MAC)
    pk = pd.concat([ben, att[ben.columns]], ignore_index=True).sort_values('ts', kind='stable')
    w = window_features(pk, keep_hosts=[ATTACKER_IP, TARGET_IP])
    is_att = w['host'] == ATTACKER_IP
    w['label'] = np.select(
        [is_att & (w['attack_tx_cur'] > 0), is_att & (w['attack_trail'] > 0), ~is_att & (w['attack_trail'] > 0)],
        [job['class'], 'post_attack', 'victim'], default='benign')
    w.insert(0, 'split', job['split'])
    w.insert(1, 'source', 'mix')
    w.insert(2, 'day', job['day'])
    w['sample_id'] = job['sample_id']
    w['scenario'], w['task_id'] = job['scenario'], job['task_id']
    w['incomplete_capture'] = job['incomplete_capture']
    w['persona'] = persona_mac or ''
    w['target_dev'] = target_mac or ''
    w.to_parquet(PART_DIR / f"{job['sample_id']}.parquet", index=False)
    a = w[w['label'] == job['class']]
    return {'sample_id': job['sample_id'], 'scenario': job['scenario'], 'task_id': job['task_id'],
            'split': job['split'], 'day': job['day'], 'persona': persona_mac or '', 'target_dev': target_mac or '',
            'bg_pkts': len(ben), 'attack_pkts': int((att['label'] == 'attack').sum()),
            'attack_win_by_count': int(a['f_w_by_count'].sum()),
            'attack_pkt_share_med': float((a['attack_tx_cur'] / a['f_w_pkts']).median()) if len(a) else np.nan,
            **{f'win_{l}': int((w['label'] == l).sum()) for l in ['benign', job['class'], 'post_attack', 'victim']},
            **{f'aug_{k}': str(v) for k, v in aug.items()}}


split_days = days.groupby('split')['day'].apply(list).to_dict()
jobs = []
for r in task.to_dict('records'):
    seed = zlib.crc32(f"{r['scenario']}/{r['task_id']}/{SEED}".encode())
    rng = np.random.default_rng(seed)
    jobs.append({'sample_id': f"{r['scenario']}_{int(r['task_id']):04d}", 'scenario': r['scenario'],
                 'task_id': r['task_id'], 'class': r['class'], 'split': r['split'],
                 'day': rng.choice(split_days[r['split']]), 'seed': seed, 'parquet': r['parquet'],
                 'attack_t0': r['attack_t0'], 'attack_t1': r['attack_t1'],
                 'incomplete_capture': r['incomplete_capture']})
    for k in range(1, DOS_AUG + 1) if r['class'] == 'dos' else []:  # biến thể DoS tăng cường, cùng split
        sid = f"{r['scenario']}_{int(r['task_id']):04d}_aug{k}"
        seed_k = zlib.crc32(f'{sid}/{SEED}'.encode())
        jobs.append({**jobs[-1], 'sample_id': sid, 'seed': seed_k, 'aug': True,
                     'day': np.random.default_rng(seed_k).choice(split_days[r['split']])})
jobs.sort(key=lambda j: -(j['attack_t1'] - j['attack_t0']))
with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    mix_res = pd.DataFrame(list(ex.map(build_sample, jobs)))
mix_res.to_csv(REPORT_DIR / '03b_mix_samples.csv', index=False)
old = pd.read_csv(REPORT_DIR / '03_mix_samples.csv').set_index('sample_id')
lab = mix_res[mix_res['sample_id'].isin(old.index)]  # mẫu DoS không có trong 03
assert len(lab) == len(old) and (old.loc[lab['sample_id'], ['day', 'persona']].fillna('').to_numpy()
                                 == lab[['day', 'persona']].fillna('').to_numpy()).all(), 'trộn khác notebook 03'
win_cols = [c for c in mix_res if c.startswith('win_')]
mix_res.groupby('scenario').agg(tasks=('sample_id', 'size'), attack_pkts=('attack_pkts', 'sum'),
                                **{c: (c, 'sum') for c in win_cols + ['attack_win_by_count']},
                                attack_pkt_share_med=('attack_pkt_share_med', 'median')).round(3)

,tasks,attack_pkts,win_benign,win_bruteforce,win_post_attack,win_victim,win_recon,win_dos,attack_win_by_count,attack_pkt_share_med
scenario,,,,,,,,,,
arp_discovery,2,970,72,0.0,10,59,20.0,0.0,18,1.000
dos_ack,64,11174542,2342,0.0,594,27923,0.0,196914.0,196852,1.000
dos_fin,60,9373028,1323,0.0,163,13757,0.0,175011.0,174954,1.000
dos_icmp,24,3231142,1587,0.0,221,15729,0.0,49399.0,49377,1.000
dos_rst,64,10556813,1307,0.0,116,1848,0.0,210692.0,210629,1.000
dos_syn,64,10663247,1423,0.0,343,25794,0.0,188728.0,188666,1.000
dos_udp,48,6415581,1216,0.0,243,947,0.0,128354.0,128313,1.000
firewall_ack_mapping,252,135777,5886,0.0,992,6479,1159.0,0.0,909,0.889
icmp_echo_discovery,118,1363,3755,0.0,1061,2317,258.0,0.0,10,1.000


## 5. Gộp, gán `use_train` và `sample_weight`, xuất CSV

In [8]:
META = ['split', 'source', 'day', 'sample_id', 'scenario', 'task_id', 'incomplete_capture', 'persona', 'target_dev',
        'host', 't_start', 't_end', 'b10', 'label', 'use_train', 'sample_weight', 'attack_tx_cur', 'attack_trail']
parts = sorted(PART_DIR.glob('*.parquet'))
allw = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
for c, v in [('scenario', ''), ('task_id', -1), ('incomplete_capture', 0), ('persona', ''), ('target_dev', '')]:
    allw[c] = allw[c].fillna(v) if c in allw else v
allw['use_train'] = allw['label'].isin(list(CLASS_WEIGHT)).astype(int)


def weights(df):
    w = pd.Series(0.0, index=df.index)
    u = df['use_train'] == 1
    att = u & (df['label'] != 'benign')
    # mỗi mẫu (task) tổng 1 -> kịch bản tổng ∝ sqrt(số cửa sổ) trong lớp -> mỗi lớp tổng CLASS_WEIGHT
    scen = df.loc[att, 'scenario']
    w[att] = 1.0 / df[att].groupby('sample_id')['label'].transform('size')
    w[att] = w[att] / w[att].groupby(scen).transform('sum') * np.sqrt(scen.map(scen.value_counts()))
    ben = u & (df['label'] == 'benign')
    w[ben] = 1.0
    tii = ben & (df['source'] == 'benign_tii')  # nền khó: TII_BENIGN_SHARE tổng trọng số benign
    if tii.any() and (ben & ~tii).any():
        w[tii] = TII_BENIGN_SHARE / (1 - TII_BENIGN_SHARE) * (ben & ~tii).sum() / tii.sum()
    masks = {cls: u & (df['label'] == cls) for cls in CLASS_WEIGHT}

    def class_norm(w):
        for cls, m in masks.items():
            if m.any():
                w[m] *= CLASS_WEIGHT[cls] / w[m].sum()
        return w

    w = class_norm(w)
    for _ in range(5):  # trần mỗi cửa sổ rồi cân lại lớp
        w = class_norm(w.clip(upper=MAX_WINDOW_SHARE * w.sum()))
    return w * (u.sum() / w.sum())


allw['sample_weight'] = 0.0
for s in ['train', 'val']:
    m = allw['split'] == s
    allw.loc[m, 'sample_weight'] = weights(allw[m])

summary = (allw.groupby(['split', 'source', 'label'])
           .agg(windows=('t_end', 'size'), share_by_count=('f_w_by_count', 'mean'),
                weight_sum=('sample_weight', 'sum')).reset_index())
summary.to_csv(REPORT_DIR / '03b_window_label_summary.csv', index=False)
summary.round(4)

,split,source,label,windows,share_by_count,weight_sum
0,test,benign_day,benign,271582,0.0683,0.0000
1,test,benign_tii,benign,8255,0.9759,0.0000
2,test,mix,benign,10675,0.2101,0.0000
3,test,mix,bruteforce,139587,0.9973,0.0000
4,test,mix,dos,91439,0.9996,0.0000
5,test,mix,post_attack,2710,0.4494,0.0000
6,test,mix,recon,11607,0.9331,0.0000
7,test,mix,victim,169673,0.9568,0.0000
8,train,benign_day,benign,787145,0.0768,432850.5523
9,train,benign_tii,benign,38569,0.9747,49720.3750


In [9]:
for s in ['train', 'val', 'test']:
    allw.loc[allw['split'] == s, META + FEATURES].to_csv(WIN_DIR / f'{s}.csv', index=False)
for p in parts:
    p.unlink()
PART_DIR.rmdir()
{s: (WIN_DIR / f'{s}.csv').stat().st_size // 2**20 for s in ['train', 'val', 'test']}  # MB

{'train': 1771, 'val': 484, 'test': 493}

In [10]:
# Kiểm tra nhanh độ tách: median một số đặc trưng theo nhãn (train)
chk_cols = ['f_w_pkts', 'f_w_dur', 'f_w_pps', 'f_w_new_pair_ratio', 'f_w_d_port', 'f_w_rx_rst_ratio',
            'f_d_port_60', 'f_d_dst_300', 'f_syn_per_pair_60', 'f_d_icmp_300']
med = allw[allw['split'] == 'train'].groupby(['label', 'scenario'])[chk_cols].median().round(3)
med.to_csv(REPORT_DIR / '03b_feature_median_by_label.csv')
med

f_w_pkts  f_w_dur   f_w_pps  f_w_new_pair_ratio  \
label  scenario                                                             
benign                         4.0   10.000     0.400                 0.0   
       arp_discovery           3.0   10.000     0.300                 0.0   
       dos_ack                29.0   10.000     2.900                 0.0   
       dos_fin                 8.0   10.000     0.800                 0.0   
       dos_icmp                4.0   10.000     0.400                 0.0   
...                            ...      ...       ...                 ...   
victim stealth_null_scan      50.0    1.421    35.187                 0.0   
       stealth_xmas_scan      50.0    1.453    34.420                 0.0   
       tcp_connect_scan       50.0    0.021  2345.216                 0.3   
       tcp_syn_scan           50.0    0.116   431.637                 0.0   
       udp_port_scan           3.0   10.000     0.300                 0.0   

                          f_w_d_port  f_w_rx_rst_ratio  f_d_port_60  \
label  scenario                                                       
benign                           1.0               0.0          1.0   
       arp_discovery             1.0               0.0          1.0   
       dos_ack                   1.0               0.0          2.0   
       dos_fin                   1.0               0.0          1.0   
       dos_icmp                  1.0               0.0          1.0   
...                              ...               ...          ...   
victim stealth_null_scan         2.0               0.0          3.0   
       stealth_xmas_scan         2.0               0.0          3.0   
       tcp_connect_scan         49.0               0.0        422.0   
       tcp_syn_scan              1.0               0.0          2.0   
       udp_port_scan             1.0               0.0          1.0   

                          f_d_dst_300  f_syn_per_pair_60  f_d_icmp_300  
label  scenario                                                         
benign                            3.0                0.0           0.0  
       arp_discovery              2.0                0.0           0.0  
       dos_ack                    8.0                0.0           0.0  
       dos_fin                    3.0                0.0           0.0  
       dos_icmp                   3.0                0.0           0.0  
...                               ...                ...           ...  
victim stealth_null_scan          8.0                0.0           0.0  
       stealth_xmas_scan          4.0                0.0           0.0  
       tcp_connect_scan           4.0                0.0           0.0  
       tcp_syn_scan               4.0                0.0           0.0  
       udp_port_scan              4.0                0.0           0.0  

[61 rows x 10 columns]